# NHTSA Data Profiling Summary

This notebook profiles the raw NHTSA datasets (`complaints`, `recalls`, `investigations`) to analyze column schemas, missing values, date ranges, and textual inconsistencies (e.g., messy make/model names) as required for the Vehicle Safety Early Warning project.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Set plot styling
plt.style.use('ggplot')
pd.set_option('display.max_rows', 100)

# Connect to DuckDB locally (Ensure you run this from the project root)
conn = duckdb.connect('data/nhtsa.duckdb')
print('Connected to DuckDB successfully.')

## 1. The `complaints` Table Profiling

In [ ]:
# Schema and Missing Values
table = 'complaints'
schema_df = conn.sql(f"PRAGMA table_info('raw.{table}')").df()
print("--- COMPLAINTS SCHEMA (Top 10 Columns) ---")
display(schema_df[['name', 'type']].head(10))

row_count = conn.sql(f"SELECT COUNT(*) FROM raw.{table}").fetchall()[0][0]
print(f"Total Rows: {row_count:,}")

null_queries = [f"SUM(CASE WHEN {col} IS NULL OR CAST({col} AS VARCHAR) = '' THEN 1 ELSE 0 END) AS {col}_nulls" for col in schema_df['name']]
null_sql = f"SELECT {', '.join(null_queries)} FROM raw.{table}"
nulls_df = conn.sql(null_sql).df().T
nulls_df.columns = ['Missing_Count']
nulls_df['Missing_Pct'] = (nulls_df['Missing_Count'] / row_count * 100)

# Plot top 15 columns with most missing values
messy_cols = nulls_df[nulls_df['Missing_Count'] > 0].sort_values(by='Missing_Pct', ascending=True).tail(15)
if not messy_cols.empty:
    messy_cols['Missing_Pct'].plot(kind='barh', figsize=(10, 6), color='steelblue')
    plt.title('Top 15 Missing Values (%) - Complaints')
    plt.xlabel('% Missing')
    plt.show()

In [ ]:
# Date Ranges (LDATE in Complaints)
date_sql = "SELECT MIN(LDATE) as min_date, MAX(LDATE) as max_date FROM raw.complaints WHERE LDATE IS NOT NULL AND LDATE != ''"
dates_df = conn.sql(date_sql).df()
print("--- COMPLAINTS DATE RANGE (LDATE) ---")
display(dates_df)

# Visualize complaints over time (by Year)
yearly_sql = """
SELECT SUBSTRING(CAST(LDATE AS VARCHAR), 1, 4) as Year, COUNT(*) as count
FROM raw.complaints
WHERE LDATE IS NOT NULL AND LDATE != '' AND SUBSTRING(CAST(LDATE AS VARCHAR), 1, 4) BETWEEN '1995' AND '2026'
GROUP BY Year ORDER BY Year
"""
yearly_df = conn.sql(yearly_sql).df()
yearly_df.set_index('Year')['count'].plot(kind='bar', figsize=(12, 5), color='darkred')
plt.title('Complaints Received by Year')
plt.ylabel('Number of Complaints')
plt.show()

In [ ]:
# Messiest Make/Model Names
messy_sql = """
SELECT MAKETXT, COUNT(DISTINCT MODELTXT) as distinct_models, COUNT(*) as total_records
FROM raw.complaints
GROUP BY MAKETXT
ORDER BY distinct_models DESC
LIMIT 10
"""
messy_makes = conn.sql(messy_sql).df()
print("--- MESSIEST STRING INCONSISTENCIES (MAKE/MODEL) ---")

messy_makes.set_index('MAKETXT')['distinct_models'].sort_values().plot(kind='barh', figsize=(10, 6), color='coral')
plt.title('Top 10 Makes with the Most Distinct Models (Inconsistency Indicator)')
plt.xlabel('Count of Distinct Model Names')
plt.show()

print("Sample of Messy Models for Top Make:")
top_make = messy_makes.iloc[0]['MAKETXT']
sample_models = conn.sql(f"SELECT MODELTXT, COUNT(*) as cnt FROM raw.complaints WHERE MAKETXT = '{top_make}' GROUP BY MODELTXT ORDER BY cnt DESC LIMIT 10").df()
display(sample_models)

## 2. The `recalls` Table Profiling

In [ ]:
table = 'recalls'
schema_df = conn.sql(f"PRAGMA table_info('raw.{table}')").df()
row_count = conn.sql(f"SELECT COUNT(*) FROM raw.{table}").fetchall()[0][0]
print(f"--- RECALLS TABLE ---\nTotal Rows: {row_count:,}")

null_queries = [f"SUM(CASE WHEN {col} IS NULL OR CAST({col} AS VARCHAR) = '' THEN 1 ELSE 0 END) AS {col}_nulls" for col in schema_df['name']]
null_sql = f"SELECT {', '.join(null_queries)} FROM raw.{table}"
nulls_df = conn.sql(null_sql).df().T
nulls_df.columns = ['Missing_Count']
nulls_df['Missing_Pct'] = (nulls_df['Missing_Count'] / row_count * 100)
print("\nTop Missing Values:")
display(nulls_df[nulls_df['Missing_Count'] > 0].sort_values(by='Missing_Pct', ascending=False).head(10))

# Recalls Date Range
date_sql = "SELECT MIN(RCDATE) as min_date, MAX(RCDATE) as max_date FROM raw.recalls WHERE RCDATE IS NOT NULL AND RCDATE != ''"
print("\nRecalls Date Range (RCDATE):")
display(conn.sql(date_sql).df())

## 3. The `investigations` Table Profiling

In [ ]:
table = 'investigations'
schema_df = conn.sql(f"PRAGMA table_info('raw.{table}')").df()
row_count = conn.sql(f"SELECT COUNT(*) FROM raw.{table}").fetchall()[0][0]
print(f"--- INVESTIGATIONS TABLE ---\nTotal Rows: {row_count:,}")

null_queries = [f"SUM(CASE WHEN {col} IS NULL OR CAST({col} AS VARCHAR) = '' THEN 1 ELSE 0 END) AS {col}_nulls" for col in schema_df['name']]
null_sql = f"SELECT {', '.join(null_queries)} FROM raw.{table}"
nulls_df = conn.sql(null_sql).df().T
nulls_df.columns = ['Missing_Count']
nulls_df['Missing_Pct'] = (nulls_df['Missing_Count'] / row_count * 100)
print("\nTop Missing Values:")
display(nulls_df[nulls_df['Missing_Count'] > 0].sort_values(by='Missing_Pct', ascending=False).head(10))

# Investigations Date Range (ODATE)
date_sql = "SELECT MIN(ODATE) as min_date, MAX(ODATE) as max_date FROM raw.investigations WHERE ODATE IS NOT NULL AND ODATE != ''"
print("\nInvestigations Date Range (ODATE):")
display(conn.sql(date_sql).df())

In [ ]:
conn.close()